# Realizando a importação das bibliotecas necessárias

In [1]:
# Importação das Bibliotecas necessárias

import pandas as pd
import os
import kagglehub

# Importação dos dados e transformação em DataFrame

In [ ]:
# Download latest version
path = kagglehub.dataset_download("namespaiva/base-varejo")

print("Path to dataset files:", path)

In [96]:
# Fazendo a importação do arquivo para dentro da variável "arquivo_csv"
# Como o arquivo esta dentro da pasta do Kaggle, é utilizada a biblioteca OS para realizar a importação
arquivo_csv = os.path.join(path, 'Base Varejo.csv')

# Lendo o arquivo CSV e realizando o encoding.
df = pd.read_csv(arquivo_csv, sep=';', encoding='latin-1')

In [97]:
# Transformando o conjunto de dados em um DataFrame
df = pd.DataFrame(df)

# Análises Básicas: Conhecendo o DataFrame

In [ ]:
# Verificando as primeiras linhas do DataFrame
df.head()

In [ ]:
# Verificando a quantidade de linhas e colunas no Dataframe
linhas, colunas = df.shape
print(f"Linhas: {linhas} | Colunas: {colunas}")

In [ ]:
# Verificando as últimas linhas do DataFrame
df.tail()

In [ ]:
# Informações adicionais sobre os dados
df.info()

In [104]:
# Gerando estatísticas descritivas básicas, referentes às colunas numéricas do DataFrame
estatisticas_descritivas = df.describe()

In [ ]:
# Imprimindo as estatísticas descritivas
estatisticas_descritivas

In [ ]:
# Soma dos valores duplicados
print(df.duplicated().sum())

# Processo de Tratamento e Limpeza dos dados

In [100]:
# Deletando as colunas que não serão utilizadas
df = df.drop(columns=['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13'])

In [98]:
# Convertendo a coluna de Data para o formato DateTime
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y')

In [99]:
'''
ESTABELECENDO O TIPO DAS DEMAIS COLUNAS DO DATAFRAME

Para as colunas categóricas, foi utilizado o tipo "category", já para os textos
comuns, como nomes e descrições, foi utilizado o tipo "String"

'''

df = df.astype({
    'CL_GENERO': 'category',
    'CL_SEG': 'category',
    'PR_CAT': 'string',
    'PR_NOME': 'string',
})

In [ ]:
'''
Análise de Duplicatas (Investigação do Sprint 3)

Verificando exemplos de linhas que são 100% idênticas no DataFrame para entender se
representam compras repetidas do mesmo produto na mesma transação ou registros duplicados acidentalmente.

'''
# Filtrando e exibindo algumas linhas que possuem duplicatas exatas para análise visual
duplicadas_completas = df[df.duplicated(keep=False)].sort_values(by=['CO_ID', 'PR_ID'])
display(duplicadas_completas.head(10))

In [ ]:
'''
Proposta de Solução: Criando a coluna de Quantidade (PR_QTD)

Em vez de excluir as duplicadas e perder a informação de vendas, os dados idênticos
serão agrupados e serão contadas quantas vezes cada produto foi levado na mesma compra.

'''

# Agrupando todas as colunas existentes e contando a ocorrência delas para gerar a Quantidade
colunas_agrupamento = list(df.columns)

# Criando um DataFrame consolidado agregando o tamanho do grupo como 'PR_QTD'
df_consolidado = df.groupby(colunas_agrupamento, as_index=False, observed=True).size().rename(columns={'size': 'PR_QTD'})

# Exibindo o novo formato dos dados e o total de linhas agora
linhas_novas, colunas_novas = df_consolidado.shape
print(f"Antes tínhamos: {df.shape[0]} linhas. Agora consolidado temos: {linhas_novas} linhas.")
df_consolidado = df_consolidado.sort_values(by=['CO_ID', 'PR_ID'])

In [111]:
# Após a organização do DataFrame com base nas colunas 'CO_ID' e 'PR_ID', o index ficou fora de ordem
# Neste caso, foi realizado um reset no Index original.
df_consolidado = df_consolidado.reset_index(drop=True)

In [ ]:
print(df_consolidado['PR_QTD'].sum() == len(df))

In [130]:
# Verificando a quantidade de valores não informados na coluna "PR_CAT"
pr_cat_nao_informado = df_consolidado['PR_CAT'].value_counts(dropna=False)

In [124]:
# Alterando '#N/D' para 'Não informado' na coluna "PR_CAT"
df_consolidado['PR_CAT'] = df_consolidado['PR_CAT'].replace('#N/D', 'Não informado')

In [ ]:
# Printando o resultado
print(df_consolidado['PR_CAT'].value_counts(dropna=False))

In [126]:
# Verificando a quantidade de valores não informados na coluna "PR_NOME"
pr_nome_nao_informado = df_consolidado['PR_NOME'].value_counts(dropna=False)

In [127]:
# Alterando '#N/D' para 'Não informado' na coluna "PR_NOME"
df_consolidado['PR_NOME'] = df_consolidado['PR_NOME'].replace('#N/D', 'Não informado')

In [ ]:
# Printando o resultado
print(df_consolidado['PR_NOME'].value_counts(dropna=False))

# INSIGHTS

In [145]:
'''
CONSIDERAÇÕES SOBRE O CÓDIGO ABAIXO

Este código busca identificar a quantidade de filhos que cada cliente possui.
Assumindo que o número de filhos é sempre o mesmo, quando o cliente preenche
os dados na compra, foi utilizado o argumento ".first()" para identificar
o primeiro registro referente à cada ID.

'''

df_num_filhos = df_consolidado.groupby('CL_ID', as_index=False)['CL_FHL'].first()

In [ ]:
'''
Validando a premissa: Cada cliente tem sempre o mesmo número de filhos

O código abaixo busca validar a premissa. Caso o resultado seja 0
o atributo ".first()" está funcionando corretamente.

'''
print((df_consolidado.groupby('CL_ID')['CL_FHL'].nunique() > 1).sum())

In [ ]:
# Printando o resultado da célula acima
display(df_num_filhos)

In [ ]:
# Analisando agrupamentos. No caso em questão, é analisado a soma total de produtos comprados por cada gênero
df_consolidado.groupby('CL_GENERO')['PR_QTD'].agg(['sum', 'mean', 'count', 'min', 'max'])